In [ ]:
"""
Simulation setup.

    run_fit(...)   build a randomised ground truth, fit a twin to data drawn
                   from it, and return both models with the training history
                   and the masks the loss was computed under

The training mechanism is the three-stage pipeline train_twin() runs: a
correspondence fit for the camera affine, gradient descent on the optics, and
then the stray-light field. The stages, their parameter groups, learning rates
and schedules are the ones train_twin() builds - nothing about the training
is redefined here.

Which stages run depends on the configuration. A fixed camera skips the
correspondence stage, and a ground truth with no stray light skips the
background stage.

Seeds. build_ground_truth(seed=...) randomises the SLM field, the stray-light
field and the six Seidel coefficients, but not the LUT scale, the crosstalk
sigmas, the deadspace reflectance, the field envelope or the camera affine:
those are arguments with fixed defaults, so seeding alone would give every
system the same LUT and the same crosstalk. They are drawn here from
TRUE_PARAM_RANGES by an RNG seeded from true_seed. Set TRUE_PARAM_RANGES to
None to fall back to the constructor defaults.

The twin's starting point is not randomised beyond its initial field: on real
hardware the fit always starts from the same nominal guess, so init_lut_scale
and the rest stay constants. twin_seed controls the twin's noisy initial field
and the batch shuffling during training.
"""
import contextlib
from dataclasses import dataclass
from typing import Optional
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.twin.model import OpticsGeometry, ModuleFlags, HoloSystem
from src.training.build_true import build_ground_truth
from src.training.build_twin import build_twin
from src.acquisition.data import make_sim_batch_fn
from src.training.train import DataSource, LossSpec
from src.training.pipeline import train_twin
from src.training.masks import get_zod_mask_checker
from src.loss.losses import build_ms_ssim_loss, mse_loss
from src.loss.reg import total_variation_reg



# ============================================================================
# GEOMETRY
# ============================================================================
# Choosing the focal length. The first diffraction order spans
# f * lambda / p_slm (mm, nm, um -> um) and the sensor spans
# n_camera * p_camera, so the fraction of the order the sensor sees is
#
#     coverage = n_camera * p_camera / (f * lambda / p_slm)
#
# which with a 200 px camera at 4 um, 500 nm and a 4 um SLM pitch is
# 6.4 / f[mm]. f = 7.5 mm gives 0.853, close to the real system's 0.872, and
# leaves margin for a drawn misalignment of up to 4 deg and 6 px.
#
# electrode_width comes down with the pitch, which it must not exceed;
# 3.9 um keeps the fill factor at (3.9/4)^2 = 0.951, as on the real device.
GEOMETRY_KWARGS = dict(
    slm_pixels=(160, 200), scale=2, device="cpu",
    slm_pixel_pitch=4.0,        # um
    electrode_width=3.9,        # um, fill factor 0.951
    wavelength=500.0,           # nm
    focal_length=7.5,           # mm
    camera_shape=(200, 200),
    camera_pixel_pitch=4.0,     # um
    fov=1, camera_affine_supersample=2,
)
GEOMETRY = OpticsGeometry(**GEOMETRY_KWARGS)

NUM_TRAIN_SAMPLES = 200
TRUE_SEED = 10
TWIN_SEED = 0


def geometry_with(**overrides) -> OpticsGeometry:
    """The same optics with a few fields replaced. Rebuilt rather than copied
    - OpticsGeometry derives its grid sizes in __post_init__."""
    return OpticsGeometry(**{**GEOMETRY_KWARGS, **overrides})


def first_order_um(geometry: OpticsGeometry = None) -> float:
    """Width of the first diffraction order, in microns."""
    g = geometry or GEOMETRY
    return g.focal_length * 1e3 * g.wavelength * 1e-3 / g.slm_pixel_pitch


# ============================================================================
# CONSTANTS
# ============================================================================
BATCH_SIZE = 4
MICRO_BATCH_SIZE = 4
ITERATIONS = 500              # optics stage
BACKGROUND_ITERATIONS = 100   # stray-light stage, when there is one
NUM_CHECKER_SAMPLES = 4       # gratings for the camera correspondence fit
NUM_VALIDATION_SAMPLES = 10
MASK_THRESHOLD = 0.1

# Validation holograms come from the same generator as training but at an
# index offset the training pool cannot reach, so the two stay disjoint
# however large num_train_samples grows.
VALIDATION_OFFSET = 10_000

TRUE_PARAM_RANGES = {
    "lut_scale": (1.5 * np.pi, 5.0 * np.pi),
    "crosstalk_sigma": (0.30, 0.70),
    "deadspace_reflectance": (-0.9, 0.9),
    # Used only when ideal_affine is False.
    "affine_rotation_deg": (-4.0, 4.0),
    "affine_scale": (0.97, 1.03),
    "affine_shift_px": (-6.0, 6.0),
    "slm_field_sigma": (0.75, 2),
}

# Seidel draw width: the coefficients come from +/- aberr_scale / 2.
ABERR_SCALE = 15.0

# The twin's starting point.
INIT_LUT_SCALE = 4 * np.pi
INIT_CROSSTALK_SIGMA = (0.3, 0.3)
INIT_DEADSPACE_REFLECTANCE = 0.8
NUM_PUPIL_TILES = 20
USE_CROSSTALK_RESIDUAL = False
USE_CROSSTALK_FAST_APPROX = False
USE_LUT_DEPTH = False

# Regularisation: evaluated, never trained on. In simulation the twin's model
# class contains the ground truth exactly and there is no unmodelled noise for
# a prior to guard against, so the gap between the two training curves is what
# the regulariser would have cost.
TV_LAMBDA_AMP = 1e-3
TV_LAMBDA_PHASE = 1e-3
TV_LAMBDA_L2 = 1e-4


@contextlib.contextmanager
def quiet_figures(quiet: bool = True):
    """
    Every stage renders its own diagnostics as it finishes (see
    TrainingStage._report). This suppresses the display without touching 
    the training code - the figures are still built, just closed 
    instead of shown.
    """
    if not quiet:
        yield
        return
    show = plt.show
    plt.show = lambda *args, **kwargs: None
    try:
        yield
    finally:
        plt.show = show
        plt.close("all")


def draw_true_params(true_seed: int) -> dict:
    """The ground-truth scalars build_ground_truth's own seed does not reach."""
    if TRUE_PARAM_RANGES is None:
        return {}
    rng = np.random.default_rng(true_seed)
    return {
        "lut_scale": float(rng.uniform(*TRUE_PARAM_RANGES["lut_scale"])),
        "crosstalk_sigma": tuple(rng.uniform(*TRUE_PARAM_RANGES["crosstalk_sigma"], size=2)),
        "deadspace_reflectance": float(rng.uniform(*TRUE_PARAM_RANGES["deadspace_reflectance"])),
        "affine_params": {
            "rotation": float(np.radians(rng.uniform(*TRUE_PARAM_RANGES["affine_rotation_deg"]))),
            "scale": tuple(rng.uniform(*TRUE_PARAM_RANGES["affine_scale"], size=2)),
            "shift": tuple(rng.uniform(*TRUE_PARAM_RANGES["affine_shift_px"], size=2)),
        },
        "slm_field_sigma": float(rng.uniform(*TRUE_PARAM_RANGES["slm_field_sigma"])),
    }


# ============================================================================
# RESULT
# ============================================================================

@dataclass
class Fit:
    """Everything the figures draw, and nothing else."""
    model_true: HoloSystem
    model_twin: HoloSystem
    histories: dict                 # {"camera"/"optics"/"background": history}
    zod_mask: torch.Tensor
    saturation_mask: torch.Tensor
    geometry: OpticsGeometry
    # what the run actually had, so a figure can decide what to draw
    use_background: bool
    fix_camera: bool
    twin_pupil: bool
    num_train_samples: int
    slm_field_scale: float
    label: str = ""

    @property
    def history(self) -> dict:
        """The optics stage - the convergence curve the figures show."""
        return self.histories["optics"]


# ============================================================================
# RUN
# ============================================================================

def run_fit(
    geometry: OpticsGeometry = None,
    num_train_samples: int = NUM_TRAIN_SAMPLES,
    true_seed: int = TRUE_SEED,
    twin_seed: int = TWIN_SEED,
    *,
    iterations: int = ITERATIONS,
    background_iterations: int = BACKGROUND_ITERATIONS,
    use_background: bool = False,
    ideal_affine: bool = True,
    fix_camera: bool = True,
    twin_pupil: bool = True,
    camera_offset: tuple = None,
    slm_field_scale: float = 1.0,
    num_scales: int = 5,
    num_validation_samples: int = NUM_VALIDATION_SAMPLES,
    batch_size: int = BATCH_SIZE,
    micro_batch_size: int = MICRO_BATCH_SIZE,
    show_stages: bool = False,
    label: str = "",
) -> Fit:
    """
    Fit a twin to data drawn from a randomised ground truth.

    Parameters
    ----------
    geometry : OpticsGeometry
        Held constant across a sweep; this is the system being simulated.
    num_train_samples : int
        Random holograms available to train on. The iteration budget is fixed
        independently of this: train_model() draws a fresh batch_size-sample
        subset per epoch rather than passing over the whole set, so a smaller
        training set is also seen more often.
    true_seed, twin_seed : int
        The ground-truth system, and the twin's starting point plus its batch
        shuffling.
    twin_pupil : bool
        Give the twin a Seidel pupil module. False leaves the ground truth's
        aberrations in place but denies the twin any way to represent them.
    camera_offset : (u, v), optional
        Move the sensor off the optical axis by this much, in units where the
        first diffraction order spans [-0.5, 0.5]. Needs a geometry built
        with a large enough affine_shift_range.
    slm_field_scale : float
        Multiplies the ground truth's SLM field, which is an exposure change
        in everything but name: the camera model carries no photon or read
        noise, so the only thing this can affect is how much of the frame
        clips at saturation.
    num_scales : int
        MS-SSIM scales in the structure loss. Five needs the shorter image
        side to exceed (11 - 1) * 2**4 = 160 px, since the mask pyramid is
        built at construction; one scale degenerates to plain masked SSIM and
        drops that floor to 11 px, which is what a sweep over small sensors
        needs. Runs at different scale counts are not comparable.
    """
    geometry = geometry or GEOMETRY
    module_flags = ModuleFlags(background=use_background, pupil=twin_pupil)

    # --- ground truth -----------------------------------------------------
    np.random.seed(true_seed)
    torch.manual_seed(true_seed)
    model_true = build_ground_truth(
        geometry,
        module_flags=ModuleFlags(background=use_background),
        use_pupil_vignetting=False,
        use_crosstalk_fast_approx=USE_CROSSTALK_FAST_APPROX,
        ideal_affine=ideal_affine,
        use_lut_depth=USE_LUT_DEPTH,
        aberr_scale=ABERR_SCALE,
        seed=true_seed,
        **draw_true_params(true_seed),
    )
    if camera_offset is not None:
        model_true.camera.shift_in_far_field(*camera_offset)
    if slm_field_scale != 1.0:
        # Through the setter, so the frozen field cache is rebuilt.
        model_true.slm_field.field = model_true.slm_field.field * slm_field_scale

    # --- data -------------------------------------------------------------
    batch_fn = make_sim_batch_fn(model_true, pattern="random")
    data_train = DataSource(batch_fn, total_samples=num_train_samples)
    data_validation = DataSource(lambda idx: batch_fn(idx + VALIDATION_OFFSET),
                                 total_samples=num_validation_samples)
    data_checker = DataSource(make_sim_batch_fn(model_true, pattern="checkerboard"),
                              total_samples=NUM_CHECKER_SAMPLES)
    data_mask = DataSource(make_sim_batch_fn(model_true, pattern="mask"),
                           total_samples=1)

    # --- twin -------------------------------------------------------------
    np.random.seed(twin_seed)
    torch.manual_seed(twin_seed)
    model_twin = build_twin(
        geometry,
        init_lut_scale=INIT_LUT_SCALE,
        init_crosstalk_sigma=INIT_CROSSTALK_SIGMA,
        num_pupil_tiles=NUM_PUPIL_TILES,
        init_deadspace_reflectance=INIT_DEADSPACE_REFLECTANCE,
        use_crosstalk_fast_approx=USE_CROSSTALK_FAST_APPROX,
        use_lut_depth=USE_LUT_DEPTH,
        use_crosstalk_residual=USE_CROSSTALK_RESIDUAL,
    )
    if fix_camera:
        model_twin.camera.affine_params = model_true.camera.affine_params
    elif camera_offset is not None:
        # The correspondence stage refines the estimate it is given rather
        # than searching the whole field, so the twin starts near the sensor.
        model_twin.camera.shift_in_far_field(*camera_offset)
    if not use_background:
        model_twin.background.disable()
    if not twin_pupil:
        model_twin.pupil.disable()

    # --- masks ------------------------------------------------------------
    # The zeroth order is static and carries a large share of the power. With
    # stray light present it has to be excluded from the structure loss, or
    # the fit spends its capacity on a region the hologram does not control
    # and the stray-light field never separates from it. A (1,1)-pitch probe
    # isolates it: an ideal pixel deflects all power away from the centre
    # under that pattern, so what is left there is the leakage.
    if use_background:
        _, C_mask = data_mask.batch_fn(torch.arange(1))
        zod_mask = get_zod_mask_checker(C_mask, threshold=MASK_THRESHOLD)
    else:
        zod_mask = torch.ones(geometry.camera_shape, device=geometry.device)

    # Pixels saturated in every training image carry no gradient, so they are
    # excluded from the stray-light loss.
    saturation_mask = torch.ones(geometry.camera_shape, dtype=torch.bool,
                                 device=geometry.device)
    for i in range(max(1, num_train_samples // batch_size)):
        _, C_batch = data_train.batch_fn(
            torch.arange(i * batch_size, (i + 1) * batch_size))
        saturation_mask &= (C_batch >= 0.99).all(dim=0)
    saturation_mask = (~saturation_mask).float()

    # --- losses -----------------------------------------------------------
    ms_ssim = build_ms_ssim_loss(mask=zod_mask, gaussian_num_scales=num_scales)
    mse = lambda I, T: mse_loss(I, T, mask=zod_mask)

    def structure(I, T):
        """The training objective: half MS-SSIM, half MSE, no regularisation."""
        return 0.5 * ms_ssim(I, T) + 0.5 * mse(I, T)

    def structure_with_reg(I, T):
        reg = total_variation_reg(
            field=model_twin.slm_field.field,
            lambda_tv_amp=TV_LAMBDA_AMP, lambda_tv_phase=TV_LAMBDA_PHASE,
            lambda_l2=TV_LAMBDA_L2,
        )
        return structure(I, T) + reg

    # The stray-light stage sees the whole frame: the zeroth order is exactly
    # what it is being asked to model, so masking it out here would remove the
    # signal rather than a distraction.
    background_loss = build_ms_ssim_loss(gaussian_num_scales=num_scales)

    eval_losses = [LossSpec(structure, name="structure"),
                   LossSpec(structure_with_reg, name="structure + TV reg")]

    # --- train ------------------------------------------------------------
    with quiet_figures(not show_stages):
        histories = train_twin(
            model_twin, data_checker, data_train,
            optics_loss=LossSpec(structure, name="structure"),
            background_loss=LossSpec(background_loss, name="structure"),
            sat_mask=saturation_mask,
            optics_eval_losses=eval_losses,
            back_eval_losses=[LossSpec(background_loss, name="structure")],
            data_validation=data_validation,
            batch_size=batch_size, micro_batch_size=micro_batch_size,
            fix_camera=fix_camera, exp=None,
            iterations=(iterations, background_iterations),
            module_flags=module_flags,
            run_stages=(not fix_camera, True, use_background),
        )

    return Fit(model_true=model_true, model_twin=model_twin, histories=histories,
               zod_mask=zod_mask, saturation_mask=saturation_mask, geometry=geometry,
               use_background=use_background, fix_camera=fix_camera,
               twin_pupil=twin_pupil, num_train_samples=num_train_samples,
               slm_field_scale=slm_field_scale, label=label)

In [ ]:
"""
Figure presentation.

A figure is a stack of framed parameter blocks and then a bottom row:

    block   the ground truth, the twin minus it, or the twin on its own
    mask    the zeroth-order mask the structure loss was computed under
            (only when there is stray light to separate from it)
    conv    convergence
    cgh     a target and the two reconstructions of it

A block is one row of equally-sized panels. Sub-panels carry a name but no
letter of their own; a light frame marks where one block ends and the next
begins.

Which panels appear follows the run. Stray light adds the reflected field, a
trainable camera adds the affine frames, a twin without a pupil module drops
the aberration panel, and none is drawn when the run did not have it.

Laid out in millimetres against a fixed 180 mm width. Colorbars are a fixed
physical thickness, so they stay matched across panels of different sizes.
Difference panels factor a common power of ten out of their ticks and into the
axis label.
"""
import numpy as np
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.axes_size import Fixed

from src.twin.model import HoloSystem, ModuleFlags, OpticsGeometry
from src.cgh.optimize import generate_holograms
from src.cgh.targets import get_target_checkerboard
from src.loss.losses import nmse_loss

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
mpl.rcParams["text.hinting"] = "none"
_MATH_FONT = "DejaVu Sans"
mpl.rcParams["mathtext.fontset"] = "custom"
mpl.rcParams["mathtext.rm"] = _MATH_FONT
mpl.rcParams["mathtext.it"] = f"{_MATH_FONT}:italic"
mpl.rcParams["mathtext.bf"] = f"{_MATH_FONT}:bold"
mpl.rcParams["mathtext.sf"] = _MATH_FONT
mpl.rcParams["mathtext.cal"] = _MATH_FONT

FIG_WIDTH_MM = 180.0
FIG_DPI = 400
FONT_SIZE = 7
CBAR_FONT_SIZE = 5.5
MM = 25.4

ACCENT_COLOR = "#FFCC01"
SECONDARY_COLOR = "#0084FF"
TRAIN_COLOR = "#222222"

CMAP_AMP = "hot"
CMAP_PHASE = "twilight"
CMAP_GRAY = "gray"
CMAP_RESIDUAL = "bwr"
CMAP_RESIDUAL_LINE = ACCENT_COLOR
GROUP_EDGE = "0.78"
INTERPOLATION = "nearest"

CBAR_THICKNESS_IN = 0.07
CBAR_PAD_IN = 0.05
CBAR_BLOCK_MM = (CBAR_THICKNESS_IN + CBAR_PAD_IN) * MM
CBAR_LABEL_MM = 6.5

DELTA = r"$\Delta$"

CGH_ITERATIONS = 300
CGH_MICRO_BATCH = 2
CGH_LR = 5e-2
CGH_WARMUP_FRAC = 0.1
TARGET_PITCH = 40
CAMERA_SATURATION = 1.0

OUTPUT_DIR = Path("analysis/output")


def fig_path(name: str) -> Path:
    """Output path for a figure, creating the output directory on first use."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return OUTPUT_DIR / f"{name}.pdf"


# ============================================================================
# LAYOUT PRIMITIVES
# ============================================================================
def _mm_axes(fig, fig_h_mm, x_mm, y_mm, w_mm, h_mm):
    return fig.add_axes([x_mm / FIG_WIDTH_MM,
                         1.0 - (y_mm + h_mm) / fig_h_mm,
                         w_mm / FIG_WIDTH_MM,
                         h_mm / fig_h_mm])


def _x_positions(widths_mm, gaps_mm, left_mm):
    xs, x = [], left_mm
    for i, w in enumerate(widths_mm):
        if i:
            x += gaps_mm[i - 1]
        xs.append(x)
        x += w
    return xs


def group_box(fig, fig_h_mm, x_mm, y_mm, w_mm, h_mm):
    fig.add_artist(FancyBboxPatch(
        (x_mm / FIG_WIDTH_MM, 1.0 - (y_mm + h_mm) / fig_h_mm),
        w_mm / FIG_WIDTH_MM, h_mm / fig_h_mm,
        boxstyle="round,pad=0,rounding_size=0.004",
        transform=fig.transFigure, facecolor="none", edgecolor=GROUP_EDGE,
        linewidth=0.6, zorder=0, clip_on=False))


def group_letter(fig, fig_h_mm, x_mm, y_mm, letter):
    fig.text(x_mm / FIG_WIDTH_MM, 1.0 - y_mm / fig_h_mm, letter,
             fontsize=FONT_SIZE + 1, fontweight="bold", va="bottom", ha="left")


def panel_name(fig, ax, text, y_fig):
    if not text:
        return
    trans = mtransforms.blended_transform_factory(ax.transAxes, fig.transFigure)
    ax.text(0, y_fig, text, transform=trans, fontsize=FONT_SIZE,
            va="bottom", ha="left")


def _scale_exponent(values) -> int:
    """Common power of ten to factor out of a set of tick labels."""
    peak = float(np.nanmax(np.abs(np.asarray(values, dtype=float))))
    if peak <= 0 or not np.isfinite(peak):
        return 0
    exponent = int(np.floor(np.log10(peak)))
    return exponent if abs(exponent) >= 2 else 0


def _with_exponent(label: str, exponent: int, sep: str = " ") -> str:
    """
    The factored-out power of ten, appended to a label.

    Colorbar labels run horizontally under a panel narrower than the label
    itself, so there `sep` is a newline: the exponent costs a line of height,
    which is available, rather than width, which is not. A rotated axis label
    runs down the side and has height to spare, so it stays on one line.
    """
    return label if not exponent else rf"{label}{sep}($\times 10^{{{exponent}}}$)"


def add_colorbar(fig, ax, im, label=None, exponent=0, side="bottom"):
    """
    A colorbar of fixed physical thickness, so it matches across panels of
    different sizes.

    `side` is "bottom" for a panel with no axis of its own, and "right" for
    one that has tick labels and an axis label underneath already - a
    colorbar appended below those would sit on top of them.
    """
    vertical = side == "right"
    div = make_axes_locatable(ax)
    div.set_anchor("N" if vertical else "N")
    cax = div.append_axes(side, size=Fixed(CBAR_THICKNESS_IN), pad=Fixed(CBAR_PAD_IN))
    cbar = fig.colorbar(im, cax=cax,
                        orientation="vertical" if vertical else "horizontal")
    cbar.ax.tick_params(labelsize=CBAR_FONT_SIZE)
    # Two ticks, not three: at these panel widths three run together.
    cbar.locator = mpl.ticker.MaxNLocator(nbins=2)
    cbar.update_ticks()
    axis = cbar.ax.yaxis if vertical else cbar.ax.xaxis
    if exponent:
        factor = 10.0 ** exponent
        axis.set_major_formatter(FuncFormatter(lambda v, _p: f"{v / factor:.1f}"))
    if label:
        # Horizontally the label is wider than the panel, so the exponent
        # goes on a second line; vertically it has the panel's height to run
        # down and stays on one.
        cbar.set_label(_with_exponent(label, exponent, sep=" " if vertical else "\n"),
                       fontsize=CBAR_FONT_SIZE, labelpad=1.5, linespacing=1.1)
    return cbar


def _decade_axis(ax):
    """Decade ticks on a log axis, widened to whole decades if the data does
    not cross one - minor-tick labels are wider than these panels."""
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10.0, numticks=8))
    ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    lo, hi = ax.get_ylim()
    if sum(lo <= t <= hi for t in ax.get_yticks()) < 2:
        ax.set_ylim(10.0 ** np.floor(np.log10(lo)), 10.0 ** np.ceil(np.log10(hi)))


# ============================================================================
# CGH CHECK
# ============================================================================
def compute_metrics(img: np.ndarray, reference: np.ndarray, mask: np.ndarray) -> dict:
    """
    NMSE and PSNR over the valid region.

    The reference is scaled to the image, never the reverse: absolute
    brightness is arbitrary here, and the image is the thing being judged.
    """
    img = np.asarray(img, dtype=np.float64)
    reference = np.asarray(reference, dtype=np.float64)
    i, r = img[mask], reference[mask]
    energy = float(np.mean(i * i))
    alpha  = float(np.mean(i * r) / energy) if energy > 0 else 1.0
    i = alpha * i

    DATA_RANGE = 1.0

    mse = float(np.mean((i - r) ** 2))
    NMSE = mse / float(np.mean(r ** 2))
    PSNR = 10 * np.log10(DATA_RANGE ** 2 / mse)      # DATA_RANGE = 1.0
    return {
        "NMSE": NMSE,
        "PSNR": float("inf") if mse == 0 else PSNR,
    }


def cgh_check(fit, *, num_holograms: int = 4, target_pitch: int = TARGET_PITCH,
              iterations: int = CGH_ITERATIONS, seed: int = 0) -> dict:
    """
    Design holograms on the twin, replay them through both models, and score
    the resulting far-field intensities against each other.

    Comparing parameters is arbitrary: errors that are correlated, or that
    fall in directions the far field does not care about, never reach the
    image. The twin-to-truth edge is the prediction error, and it is the
    simulated stand-in for the twin-versus-camera comparison in the
    experiment. Target-versus-anything is limited by how well a handful of
    holograms can render a checkerboard, which is a property of the CGH
    method, not of the twin.

    Everything happens on the far field, with the camera off. The camera is a
    measurement artefact rather than part of the optics a hologram is
    designed for, and leaving it out shows whether the fit predicts the
    intensity outside the sensor's own field of view and at the model's full
    resolution - which is the question a simulated system can answer and a
    real one cannot. It also makes the number comparable between runs whose
    sensors differ.

    Stray light is switched off for the DESIGN and back on for the REPLAY. A
    hologram cannot correct for it and is not designed against it, but it is
    there when the light lands.

    Both projections are then clipped at CAMERA_SATURATION. The zeroth order
    reaches tens of times the level the sensor can record, so every training
    image saw it clipped and the fit had no information about its true
    height: the twin's value there is free to be anything above saturation
    and still match the data exactly. Left unclipped it dominates both the
    picture and the metric with the one quantity that was never measured.
    """
    model_true, model_twin = fit.model_true, fit.model_twin
    geometry = fit.geometry
    was_active = {m: (m.camera.is_active, m.background.is_active)
                  for m in (model_true, model_twin)}
    target = get_target_checkerboard(geometry.far_fov_samples, pitch=target_pitch,
                                     device=geometry.device)
    try:
        for model in (model_true, model_twin):
            model.set_unfrozen(default=False)
            model.set_active(default=None, flags=ModuleFlags(camera=False, background=False))

        torch.manual_seed(seed)
        np.random.seed(seed)
        g_optim, cgh_loss = generate_holograms(
            model_twin, target, nmse_loss,
            num_epochs=iterations, batch_size=num_holograms,
            micro_batch_size=CGH_MICRO_BATCH, lr=CGH_LR,
            lr_warmup_frac=CGH_WARMUP_FRAC, mask_centre=False,
        )

        # The stray light belongs to the system, so it comes back for the
        # replay even though the hologram was not designed against it.
        for model, (_camera_on, background_on) in was_active.items():
            model.set_active(default=None,
                             flags=ModuleFlags(camera=False, background=background_on))

        def project(model):
            frames = []
            with torch.no_grad():
                for i in range(0, g_optim.shape[0], CGH_MICRO_BATCH):
                    frames.append(model(g_optim[i:i + CGH_MICRO_BATCH]))
            return torch.cat(frames).mean(dim=0).cpu().numpy()

        raw_twin = project(model_twin)
        raw_true = project(model_true)
        image_twin = np.clip(raw_twin, 0.0, CAMERA_SATURATION)
        image_true = np.clip(raw_true, 0.0, CAMERA_SATURATION)
        image_target = target.cpu().numpy()
    finally:
        for model, (camera_on, background_on) in was_active.items():
            model.set_active(default=None,
                             flags=ModuleFlags(camera=camera_on, background=background_on))

    # Scored over the whole frame: nothing is excluded from the objective, so
    # nothing is excluded from the score either.
    mask = np.ones(image_twin.shape, dtype=bool)
    return {
        "target": image_target,
        "twin": image_twin,
        "truth": image_true,
        "mask": mask,
        "cgh_loss": cgh_loss,
        "twin_vs_target": compute_metrics(image_twin, image_target, mask),
        "truth_vs_target": compute_metrics(image_true, image_target, mask),
        "truth_vs_twin": compute_metrics(image_true, image_twin, mask),
        # How much of each frame ran past what a sensor could record. A large
        # difference between the two is the unobservable zeroth order, not a
        # failure of the fit.
        "saturated_twin": float((raw_twin > CAMERA_SATURATION).mean()),
        "saturated_truth": float((raw_true > CAMERA_SATURATION).mean()),
        "peak_twin": float(raw_twin.max()),
        "peak_truth": float(raw_true.max()),
    }


def _fmt(m: dict) -> str:
    # NMSE is fixed-point when it is small and exponential when it is not; a
    # failed reconstruction can reach several thousand, which at three
    # decimals is wider than the panel.
    return f"NMSE {m['NMSE']:.3g}, PSNR {m['PSNR']:.1f} dB"


# ============================================================================
# PANEL CONTENT
# ============================================================================
SEIDEL_SHORT = ("Coma", "Ast.", "Field", "Dist.", "X-tilt", "Y-tilt")


def report_rows(fit, *, pupil: bool = True, camera: bool = True):
    """
    The panels of one system, as a single row.

    Everything a block reports sits on one line, at whatever width that
    leaves. The images are the small ones - a crosstalk kernel or an
    envelope is a handful of numbers - and giving them a quarter of the page
    each says they matter more than the plots beside them.

    The fields follow the crosstalk, images together and plots at the end, so
    the row needs as few of the wide axis gaps as possible.

    `pupil` is False for a twin built without a Seidel module: it has no
    aberration coefficients to report, and its initial ones would be a
    reading of a module that never ran.
    """
    row = [("lut", "LUT", "plot"),
           ("crosstalk", "Crosstalk", "image"),
           ("slm_amp", "SLM field", "image"),
           ("slm_phase", None, "image")]
    if fit.use_background:
        row += [("back_amp", "Reflections", "image"),
                ("back_phase", None, "image")]
    row.append(("envelope", "Envelope", "image"))
    if pupil:
        row.append(("seidel", "Aberrations", "plot"))
    if camera:
        row.append(("camera", "Camera", "plot"))
    return [row]


# A plot needs room for its tick labels and axis label; an image needs a
# little air. Both are tighter than they were when a row held five panels.
GAP_BEFORE = {"plot": 12.5, "image": 4.0}


def _row_gaps(row):
    return [GAP_BEFORE[kind] for _key, _name, kind in row[1:]]


def _block_geometry(rows, left, right):
    """
    One panel width for the whole block, and the left edges of each row.

    Every panel in a block is the same square, so the width is the smallest
    any row can afford; rows that then have width to spare are centred rather
    than left with a ragged margin.
    """
    width = min((right - left - sum(_row_gaps(row))) / len(row) for row in rows)
    positions = []
    for row in rows:
        gaps = _row_gaps(row)
        span = len(row) * width + sum(gaps)
        start = left + 0.5 * (right - left - span)
        positions.append(_x_positions([width] * len(row), gaps, start))
    return width, positions


def _draw_lut(ax, model, other):
    g = torch.linspace(0, 1, 256, device=model.geometry.device)
    with torch.no_grad():
        phi = model.lut.phase(g).cpu().numpy()
    g_np = g.cpu().numpy()
    if other is None:
        ax.plot(g_np, phi, lw=1.0, color="0.15")
        ax.set_ylabel("Phase (rad)", fontsize=FONT_SIZE)
    else:
        # phase_nodes[0] is exactly zero in both systems, so the two curves
        # share a gauge and a plain difference is the honest comparison.
        with torch.no_grad():
            phi_b = other.lut.phase(g.to(other.geometry.device)).cpu().numpy()
        delta = phi - phi_b
        exponent = _scale_exponent(delta)
        ax.plot(g_np, delta / 10.0 ** exponent, lw=1.0, color=CMAP_RESIDUAL_LINE)
        ax.axhline(0.0, color="0.6", lw=0.5, ls="--")
        ax.set_ylabel(_with_exponent(f"{DELTA} Phase (rad)", exponent),
                      fontsize=FONT_SIZE)
    ax.set_xlabel("Grayscale", fontsize=FONT_SIZE)
    ax.set_xlim(0, 1)


def _draw_seidel(ax, model, other):
    coeffs = model.pupil.coeffs.detach().cpu().numpy()
    x = np.arange(len(coeffs))
    if other is None:
        ax.bar(x, coeffs, color=SECONDARY_COLOR)
        ax.set_ylabel("Value (rad)", fontsize=FONT_SIZE)
    else:
        delta = coeffs - other.pupil.coeffs.detach().cpu().numpy()
        exponent = _scale_exponent(delta)
        ax.bar(x, delta / 10.0 ** exponent, color=CMAP_RESIDUAL_LINE)
        ax.axhline(0.0, color="0.6", lw=0.5, ls="--")
        ax.set_ylabel(_with_exponent(f"{DELTA} Value (rad)", exponent),
                      fontsize=FONT_SIZE)
    ax.set_xticks(x)
    ax.set_xticklabels(SEIDEL_SHORT, rotation=90, ha="center", va="top",
                       fontsize=FONT_SIZE - 1.5)


def _draw_camera(ax, model, other):
    """
    The sensor's footprint on the far field. The difference panel overlays the
    two frames rather than subtracting them: two rectangles on one axis say
    where the fitted camera sits relative to the true one, which a subtraction
    would destroy.
    """
    corners_cam, corners_fov = model.camera.get_corners()
    fov = corners_fov.detach().cpu().numpy()
    cam = corners_cam.detach().cpu().numpy()
    ax.plot(fov[:, 0], fov[:, 1], color="0.55", lw=0.7, ls="--", label="Far field")
    if other is None:
        ax.plot(cam[:, 0], cam[:, 1], color="0.1", lw=1.0, label="Camera")
    else:
        truth = other.camera.get_corners()[0].detach().cpu().numpy()
        # The fitted frame is drawn thinner and on top, so a good fit shows
        # as yellow inside black rather than hiding one line under the other.
        ax.plot(truth[:, 0], truth[:, 1], color="0.1", lw=1.6, label="True")
        ax.plot(cam[:, 0], cam[:, 1], color=CMAP_RESIDUAL_LINE, lw=0.8,
                label="Fitted")
    ax.invert_yaxis()
    ax.set_xlabel("u (norm)", fontsize=FONT_SIZE)
    ax.set_ylabel("v (norm)", fontsize=FONT_SIZE, labelpad=-1)
    if other is not None:
        # The frames fill the panel, so the key goes underneath it; in the
        # ground-truth row there is only one frame and none is needed.
        ax.legend(frameon=False, fontsize=FONT_SIZE - 2.5, ncol=3,
                  loc="upper right", bbox_to_anchor=(1.0, -0.72),
                  handlelength=1.0, handletextpad=0.3, columnspacing=0.8,
                  borderaxespad=0.0)


def _image(ax, data, cmap, label, other_data=None, is_phase=False,
           weights=None, extent=None, origin=None):
    kwargs = {} if extent is None else dict(extent=extent, aspect="equal")
    if origin is not None:
        kwargs["origin"] = origin
    note, exponent = None, 0
    if other_data is None:
        # No vmin/vmax: every quantity is shown over its own range. Clamping
        # an amplitude to a fixed span washes it out.
        im = ax.imshow(data, cmap=cmap, interpolation=INTERPOLATION, **kwargs)
    else:
        if is_phase:
            # A global phase leaves the far-field intensity unchanged, so a
            # raw wrapped subtraction reports a gauge as error.
            data, offset = HoloSystem._phase_diff(data, other_data,
                                                  *(weights or (None, None)))
            note = f"global {offset:+.2f} rad"
        else:
            data = data - other_data
        vmax = float(np.abs(data).max()) or 1e-12
        exponent = _scale_exponent([vmax])
        im = ax.imshow(data, cmap=CMAP_RESIDUAL, interpolation=INTERPOLATION,
                       vmin=-vmax, vmax=vmax, **kwargs)
        label = f"{DELTA} {label.replace(' (a.u.)', '')}"
    ax.set_xticks([]); ax.set_yticks([])
    if note:
        ax.text(0.03, 0.04, note, transform=ax.transAxes, ha="left", va="bottom",
                fontsize=FONT_SIZE - 3, color="0.2",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.8))
    return im, label, exponent


def _complex_field(model, key):
    if key.startswith("slm"):
        return model.slm_field.field.detach()
    return model.background.get_filtered(mode="far").detach()


def _draw_field(fig, ax, key, model, other):
    field = _complex_field(model, key)
    other_field = None if other is None else _complex_field(other, key)
    # The reflected field lives on the far-field grid, whose sample spacing
    # differs between the axes; a unit extent shows it on the square it
    # physically occupies rather than on its sample counts.
    extent = (0, 1, 0, 1) if key.startswith("back") else None
    if key.endswith("amp"):
        im, label, exponent = _image(
            ax, field.abs().cpu().numpy(), CMAP_AMP, "Amplitude (a.u.)",
            None if other is None else other_field.abs().cpu().numpy(),
            extent=extent)
    else:
        weights = None if other is None else (
            field.abs().cpu().numpy(), other_field.abs().cpu().numpy())
        im, label, exponent = _image(
            ax, field.angle().cpu().numpy(), CMAP_PHASE, "Phase (rad)",
            None if other is None else other_field.angle().cpu().numpy(),
            is_phase=True, weights=weights, extent=extent)
    add_colorbar(fig, ax, im, label, exponent)


def _draw_single(fig, ax, key, model, other):
    if key.startswith(("slm_", "back_")):
        _draw_field(fig, ax, key, model, other)
        return
    if key == "lut":
        _draw_lut(ax, model, other)
        return
    if key == "seidel":
        _draw_seidel(ax, model, other)
        return
    if key == "camera":
        _draw_camera(ax, model, other)
        return
    if key == "crosstalk":
        get = lambda m: m.pixel.kernel.detach().cpu().numpy()
        grid_x = model.pixel._grid_X.detach().cpu().numpy()
        grid_y = model.pixel._grid_Y.detach().cpu().numpy()
        im, label, exponent = _image(
            ax, get(model), CMAP_AMP, "Magnitude (a.u.)",
            None if other is None else get(other),
            extent=(grid_x.min(), grid_x.max(), grid_y.max(), grid_y.min()),
            origin="upper")
        if other is None:
            for line in (ax.axvline, ax.axhline):
                line(-0.5, color=SECONDARY_COLOR, linestyle="--", lw=0.8)
                line(0.5, color=SECONDARY_COLOR, linestyle="--", lw=0.8)
        add_colorbar(fig, ax, im, label, exponent)
        return
    if key == "envelope":
        get = lambda m: m.pixel.aperture_envelope.detach().abs().square().cpu().numpy()
        im, label, exponent = _image(ax, get(model), CMAP_GRAY, "Intensity (a.u.)",
                                     None if other is None else get(other),
                                     extent=(0, 1, 0, 1))
        add_colorbar(fig, ax, im, label, exponent)
        return
    raise KeyError(key)


# ============================================================================
# THE FIGURE
# ============================================================================
PAD_MM = 1.5
EDGE_MM = 1.5
LETTER_MM = 4.0
NAME_MM = 3.4
STACK_HALF_MM = 10.0
# The arrays are 4:5, so a row of nothing but fields is given a box of that
# shape rather than a square one that wastes a fifth of its height.
FIELD_ASPECT = 0.8
# A row of images needs room for a colorbar's ticks and label; a row with a
# plot in it needs room for rotated tick labels and an axis label as well.
REPORT_BELOW_IMAGE_MM = 6.5
REPORT_BELOW_PLOT_MM = 9.5
GROUP_GAP_MM = 3.0
BOTTOM_PANEL_MM = 25.0
BOTTOM_BELOW_MM = 8.0
CONV_AXIS_ROOM_MM = 12.0    # y-label and tick labels, left of the plot itself
BOTTOM_GAP_MM = 5.0
CGH_GAP_MM = 3.5

BLOCK_LEFT_MM = EDGE_MM + PAD_MM + 10.5
BLOCK_RIGHT_MM = FIG_WIDTH_MM - EDGE_MM - PAD_MM


def _block_rows(fit, mode):
    """
    Rows for one block. `mode` is "truth" (the ground truth on its own),
    "diff" (twin minus truth), or "twin" (the recovered twin on its own,
    for a run whose ground truth is not being shown).

    A camera held at the truth has nothing to say in a difference block, so
    its column is dropped there rather than left blank.
    """
    pupil = True if mode == "truth" else fit.twin_pupil
    camera = not (mode == "diff" and fit.fix_camera)
    return report_rows(fit, pupil=pupil, camera=camera)


def _row_metrics(row, panel_w):
    """Panel height for one row, and the space its labels need underneath."""
    fields_only = all(key.startswith(("slm_", "back_")) for key, _n, _k in row)
    height = panel_w * FIELD_ASPECT if fields_only else panel_w
    below = (REPORT_BELOW_PLOT_MM if any(kind == "plot" for _k, _n, kind in row)
             else REPORT_BELOW_IMAGE_MM)
    return height, below


def _report_height(fit, mode):
    rows = _block_rows(fit, mode)
    panel_w, _positions = _block_geometry(rows, BLOCK_LEFT_MM, BLOCK_RIGHT_MM)
    content = sum(NAME_MM + height + CBAR_BLOCK_MM + below
                  for height, below in (_row_metrics(row, panel_w) for row in rows))
    return LETTER_MM + PAD_MM + content + PAD_MM


def _report_group(fig, fig_h_mm, y_top, letter, fit, mode):
    """
    One framed block. Returns its total height in mm, letter band included.

    Every panel is square and the same square, so the block reads as one set
    of shapes rather than a mix of sizes.
    """
    box_h = PAD_MM + _block_content_height(fit, mode) + PAD_MM
    box_y = y_top + LETTER_MM
    group_box(fig, fig_h_mm, EDGE_MM, box_y, FIG_WIDTH_MM - 2 * EDGE_MM, box_h)
    group_letter(fig, fig_h_mm, EDGE_MM, box_y - 0.8, letter)
    draw_block_rows(fig, fig_h_mm, box_y + PAD_MM, fit, mode)
    return LETTER_MM + box_h


def _block_content_height(fit, mode) -> float:
    rows = _block_rows(fit, mode)
    panel_w, _positions = _block_geometry(rows, BLOCK_LEFT_MM, BLOCK_RIGHT_MM)
    return sum(NAME_MM + height + CBAR_BLOCK_MM + below
               for height, below in (_row_metrics(row, panel_w) for row in rows))


def draw_block_rows(fig, fig_h_mm, y, fit, mode) -> float:
    """The panels of one block, with no frame and no letter of their own.
    Returns the height they occupy."""
    model = fit.model_true if mode == "truth" else fit.model_twin
    other = fit.model_true if mode == "diff" else None
    rows = _block_rows(fit, mode)
    panel_w, row_positions = _block_geometry(rows, BLOCK_LEFT_MM, BLOCK_RIGHT_MM)
    metrics = [_row_metrics(row, panel_w) for row in rows]

    top = y
    for row, xs, (row_h, below) in zip(rows, row_positions, metrics):
        panel_y = y + NAME_MM
        name_y = 1.0 - (panel_y - 0.8) / fig_h_mm
        for (key, name, kind), x in zip(row, xs):
            height = row_h + (0.0 if kind == "plot" else CBAR_BLOCK_MM)
            ax = _mm_axes(fig, fig_h_mm, x, panel_y, panel_w, height)
            ax.tick_params(labelsize=FONT_SIZE - 1)
            _draw_single(fig, ax, key, model, other)
            panel_name(fig, ax, name, name_y)
        y = panel_y + row_h + CBAR_BLOCK_MM + below
    return y - top


def _mask_panel(fig, fig_h_mm, x, y_top, letter, fit):
    letter_y = y_top + LETTER_MM
    group_letter(fig, fig_h_mm, x, letter_y - 0.8, letter)
    panel_y = letter_y + PAD_MM + NAME_MM
    ax = _mm_axes(fig, fig_h_mm, x, panel_y, BOTTOM_PANEL_MM, BOTTOM_PANEL_MM)
    ax.imshow(fit.zod_mask.detach().cpu().numpy(), cmap=CMAP_GRAY,
              vmin=0, vmax=1, interpolation=INTERPOLATION)
    ax.set_xticks([]); ax.set_yticks([])
    panel_name(fig, ax, "Loss mask", 1.0 - (panel_y - 0.8) / fig_h_mm)
    ax.set_xlabel(f"{fit.zod_mask.mean():.0%} of the frame", fontsize=FONT_SIZE - 1)


def _convergence_panel(fig, fig_h_mm, x, y_top, letter, history, width):
    """`letter` is None for a panel drawn inside a group that already carries
    one, in which case no letter band is reserved above it."""
    letter_y = y_top + (LETTER_MM if letter else 0.0)
    if letter:
        group_letter(fig, fig_h_mm, x, letter_y - 0.8, letter)
    panel_y = letter_y + PAD_MM + NAME_MM
    ax = _mm_axes(fig, fig_h_mm, x + CONV_AXIS_ROOM_MM, panel_y, width,
                  BOTTOM_PANEL_MM)
    steps = np.arange(len(history["eval: structure"]))
    if "validation: structure (min)" in history:
        ax.fill_between(steps, history["validation: structure (min)"],
                        history["validation: structure (max)"],
                        color=SECONDARY_COLOR, alpha=0.35, lw=0,
                        label="validation (min-max)")
    ax.plot(steps, history["eval: structure"], lw=0.9, color=TRAIN_COLOR, label="train")
    if "eval: structure + TV reg" in history:
        ax.plot(steps, history["eval: structure + TV reg"], lw=0.9,
                color=TRAIN_COLOR, ls="--", alpha=0.55, label="train + TV reg")
    ax.set_xlabel("Epoch", fontsize=FONT_SIZE)
    ax.set_ylabel("Loss", fontsize=FONT_SIZE)
    ax.tick_params(labelsize=FONT_SIZE - 1)
    _decade_axis(ax)
    ax.legend(frameon=False, fontsize=FONT_SIZE - 1, loc="upper right",
              handlelength=1.6, handletextpad=0.5, labelspacing=0.25,
              borderaxespad=0.2)
    ax.grid(lw=0.3, color="0.92")
    ax.set_axisbelow(True)
    panel_name(fig, ax, "Convergence", 1.0 - (panel_y - 0.8) / fig_h_mm)


CGH_GROUP_W_MM = 3 * BOTTOM_PANEL_MM + 2 * CGH_GAP_MM + 2 * PAD_MM


def _cgh_group(fig, fig_h_mm, x, y_top, letter, cgh):
    """
    Far-field intensity, not camera images: the whole field at the model's
    own resolution, so the comparison covers what the sensor never saw.

    `letter` is None when this sits inside a group that already has a frame
    and a letter, in which case it draws neither of its own.
    """
    box_y = y_top + (LETTER_MM if letter else 0.0)
    box_h = PAD_MM + NAME_MM + BOTTOM_PANEL_MM + BOTTOM_BELOW_MM + PAD_MM
    if letter:
        group_box(fig, fig_h_mm, x, box_y, CGH_GROUP_W_MM, box_h)
        group_letter(fig, fig_h_mm, x, box_y - 0.8, letter)

    panel_y = box_y + PAD_MM + NAME_MM
    name_y = 1.0 - (panel_y - 0.8) / fig_h_mm
    entries = [("Target", cgh["target"], None),
               ("Twin", cgh["twin"], cgh["twin_vs_target"]),
               ("Ground truth", cgh["truth"], cgh["truth_vs_target"])]
    # Each image is normalised to its own mean and then all three are shown
    # on the target's scale. A percentile stretch would be set by the zeroth
    # order, which is a single bright point carrying a large share of the
    # power, and everything else would render black.
    shown = [img / img.mean() for _, img, _ in entries]
    vmax = float(shown[0].max())

    for k, ((name, _img, metrics), image) in enumerate(zip(entries, shown)):
        px = x + PAD_MM + k * (BOTTOM_PANEL_MM + CGH_GAP_MM)
        ax = _mm_axes(fig, fig_h_mm, px, panel_y, BOTTOM_PANEL_MM, BOTTOM_PANEL_MM)
        # The far-field grid samples a square field at different rates along
        # the two axes, so a unit extent shows it on the square it physically
        # occupies rather than on its sample counts.
        ax.imshow(image, cmap=CMAP_GRAY, vmin=0, vmax=vmax,
                  interpolation=INTERPOLATION, extent=(0, 1, 0, 1),
                  aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        panel_name(fig, ax, name, name_y)
        if metrics is not None:
            ax.set_xlabel(_fmt(metrics), fontsize=FONT_SIZE - 1.5)


def figure_system(fit, cgh, blocks=("truth", "diff"), path: str = None) -> plt.Figure:
    """
    One run, as one 180 mm figure: the requested parameter blocks stacked
    above a row of mask (when there is one), convergence, and projection.

    `blocks` chooses what is reported above that row - ("truth", "diff") for
    the first figure, ("diff",) once the ground truth has already been shown,
    ("twin",) when the twin's own parameters are the subject.
    """
    heights = [_report_height(fit, mode) for mode in blocks]
    bottom_h = LETTER_MM + PAD_MM + NAME_MM + BOTTOM_PANEL_MM + BOTTOM_BELOW_MM + PAD_MM
    fig_h_mm = EDGE_MM + sum(h + GROUP_GAP_MM for h in heights) + bottom_h + EDGE_MM

    fig = plt.figure(figsize=(FIG_WIDTH_MM / MM, fig_h_mm / MM), dpi=FIG_DPI)

    letters = iter("abcdefg")
    y = EDGE_MM
    for mode, height in zip(blocks, heights):
        _report_group(fig, fig_h_mm, y, next(letters), fit, mode)
        y += height + GROUP_GAP_MM

    cgh_x = FIG_WIDTH_MM - EDGE_MM - CGH_GROUP_W_MM
    x = EDGE_MM
    if fit.use_background:
        _mask_panel(fig, fig_h_mm, x, y, next(letters), fit)
        x += BOTTOM_PANEL_MM + BOTTOM_GAP_MM
    # The convergence plot takes whatever is left between here and the
    # projection group, rather than a fixed width that either overruns it or
    # leaves the row half empty.
    conv_width = cgh_x - (x + CONV_AXIS_ROOM_MM) - BOTTOM_GAP_MM
    _convergence_panel(fig, fig_h_mm, x, y, next(letters), fit.history, conv_width)
    _cgh_group(fig, fig_h_mm, cgh_x, y, next(letters), cgh)

    print(f"for the caption - twin to ground truth: {_fmt(cgh['truth_vs_twin'])}")
    print(f"                   height {fig_h_mm:.1f} mm, "
          f"saturated twin {cgh['saturated_twin']:.2%} / truth {cgh['saturated_truth']:.2%}")

    if path:
        # A tight bbox re-crops the canvas, so the figure would stop being
        # 180 mm wide and every panel inside it would rescale.
        fig.savefig(fig_path(path), dpi=FIG_DPI)
        print(f"Saved: {fig_path(path)}")
    return fig


# ============================================================================
# TWO RUNS IN ONE FIGURE
# ============================================================================
def _run_group(fig, fig_h_mm, y_top, letter, fit, cgh) -> float:
    """
    One run inside a single frame: its recovered parameters, its convergence
    and its projection, under one letter. Used where two runs are being
    compared and the comparison, not the panel, is what carries the letters.
    """
    rows_h = _block_content_height(fit, "twin")
    bottom_h = NAME_MM + BOTTOM_PANEL_MM + BOTTOM_BELOW_MM
    box_h = PAD_MM + rows_h + bottom_h + PAD_MM
    box_y = y_top + LETTER_MM

    group_box(fig, fig_h_mm, EDGE_MM, box_y, FIG_WIDTH_MM - 2 * EDGE_MM, box_h)
    group_letter(fig, fig_h_mm, EDGE_MM, box_y - 0.8, letter)

    y = box_y + PAD_MM
    y += draw_block_rows(fig, fig_h_mm, y, fit, "twin")

    cgh_x = FIG_WIDTH_MM - EDGE_MM - PAD_MM - CGH_GROUP_W_MM
    x = EDGE_MM + PAD_MM
    conv_width = cgh_x - (x + CONV_AXIS_ROOM_MM) - BOTTOM_GAP_MM
    _convergence_panel(fig, fig_h_mm, x, y, None, fit.history, conv_width)
    _cgh_group(fig, fig_h_mm, cgh_x, y, None, cgh)
    return LETTER_MM + box_h


def figure_runs(entries, path: str = None) -> plt.Figure:
    """
    Several runs stacked, one framed group each: parameters, convergence and
    projection together under a single letter per run.

    `entries` is [(fit, cgh), ...] in the order they should appear.
    """
    heights = [LETTER_MM + PAD_MM + _block_content_height(fit, "twin")
               + NAME_MM + BOTTOM_PANEL_MM + BOTTOM_BELOW_MM + PAD_MM
               for fit, _cgh in entries]
    fig_h_mm = EDGE_MM + sum(h + GROUP_GAP_MM for h in heights) - GROUP_GAP_MM + EDGE_MM

    fig = plt.figure(figsize=(FIG_WIDTH_MM / MM, fig_h_mm / MM), dpi=FIG_DPI)
    letters = iter("abcdefg")
    y = EDGE_MM
    for (fit, cgh), height in zip(entries, heights):
        _run_group(fig, fig_h_mm, y, next(letters), fit, cgh)
        y += height + GROUP_GAP_MM
        print(f"  {fit.label or 'run'}: twin to ground truth {_fmt(cgh['truth_vs_twin'])}")

    print(f"height {fig_h_mm:.1f} mm")
    if path:
        fig.savefig(fig_path(path), dpi=FIG_DPI)
        print(f"Saved: {fig_path(path)}")
    return fig


# ============================================================================
# THE STATISTICS FIGURE
# ============================================================================
FIG2_PANEL_MM = 36.0
FIG2_BELOW_MM = 11.0
FIG2_LEFT_MM = 12.0
FIG2_WIDTHS_MM = (40.0, 40.0, 32.0)
FIG2_GAPS_MM = (16.0, 18.0)
CBAR_WIDTH_MM = 2.2
CBAR_OFFSET_MM = 1.5    # heatmap to colorbar; its tick labels and label
                        # need the rest of the right margin

SAMPLE_CMAP = "viridis"
GRID_CMAP = "hot_r"


def _series_panel(ax, records, key, legend_title, fmt="{:g}"):
    """
    Held-out loss against epoch, one curve per run, coloured in the order the
    runs were swept so the key reads as a scale rather than a set of
    unrelated colours. The band is the min-max across the validation
    holograms, which is what says whether a curve is converged or just
    averaging over a spread.
    """
    def sweep_color(index, total):
        blue = np.array(mpl.colors.to_rgb(SECONDARY_COLOR))
        return tuple(blue * index / (total - 1) + np.array(mpl.colors.to_rgb(ACCENT_COLOR)) * (1.0 - index / (total - 1)))

    records = sorted(records, key=lambda r: r[key])
    for k, record in enumerate(records):
        steps = np.arange(len(record["train"]))

        colour = sweep_color(k, len(records))
        ax.fill_between(steps, record["validation_min"], record["validation_max"],
                        color=colour, alpha=0.20, lw=0)
        ax.plot(steps, record["train"], lw=0.9, color=colour,
                label=fmt.format(record[key]))
    ax.set_xlabel("Epoch", fontsize=FONT_SIZE)
    ax.set_ylabel("Loss", fontsize=FONT_SIZE)
    _decade_axis(ax)
    legend = ax.legend(frameon=False, fontsize=FONT_SIZE - 1, loc="lower left",
                       title=legend_title, handlelength=1.2, handletextpad=0.5,
                       labelspacing=0.2, borderaxespad=0.2, ncol=2,
                       columnspacing=0.8)
    legend.get_title().set_fontsize(FONT_SIZE - 1)
    ax.grid(lw=0.3, color="0.92")
    ax.set_axisbelow(True)


def figure_statistics(study, path: str = None) -> plt.Figure:
    """
    Three sweeps of the same ground truth: how much data the fit needs, how
    much exposure it tolerates, and how little of the field it can be given.

        a  held-out loss against training-set size
        b  held-out loss against exposure
        c  the geometry sweep: how much of the field the sensor sees against
           how coarsely it samples it
    """
    fig_h_mm = EDGE_MM + LETTER_MM + NAME_MM + FIG2_PANEL_MM + FIG2_BELOW_MM + EDGE_MM
    fig = plt.figure(figsize=(FIG_WIDTH_MM / MM, fig_h_mm / MM), dpi=FIG_DPI)
    xs = _x_positions(FIG2_WIDTHS_MM, FIG2_GAPS_MM, FIG2_LEFT_MM)
    letter_y = EDGE_MM + LETTER_MM
    panel_y = letter_y + NAME_MM
    name_y = 1.0 - (panel_y - 0.8) / fig_h_mm

    def new_panel(k, letter, name, width=None):
        group_letter(fig, fig_h_mm, max(xs[k] - 11.0, EDGE_MM), letter_y - 0.8, letter)
        ax = _mm_axes(fig, fig_h_mm, xs[k], panel_y,
                      FIG2_WIDTHS_MM[k] if width is None else width, FIG2_PANEL_MM)
        ax.tick_params(labelsize=FONT_SIZE - 1)
        panel_name(fig, ax, name, name_y)
        return ax

    # --- (a) how much data ------------------------------------------------
    _series_panel(new_panel(0, "a", "Training-set size"),
                  study["samples"], "num_train_samples", "pairs")

    # --- (b) how much exposure --------------------------------------------
    _series_panel(new_panel(1, "b", "Exposure"),
                  study["exposures"], "exposure", r"$\times$ nominal")

    # --- (c) the geometry sweep -------------------------------------------
    ax = new_panel(2, "c", "Sensor geometry")
    crops = sorted({cell["crop"] for cell in study["grid"]})
    scales = sorted({cell["pixel_scale"] for cell in study["grid"]})
    values = np.full((len(crops), len(scales)), np.nan)
    for cell in study["grid"]:
        if cell.get("projection") is not None:
            values[crops.index(cell["crop"]), scales.index(cell["pixel_scale"])] = \
                cell["projection"]

    # Decade limits on the colour scale. Over a range narrower than one
    # decade matplotlib labels the bar in scientific notation, which is wider
    # than the margin and gets clipped.
    finite = values[np.isfinite(values)]
    norm = None
    if finite.size:
        lo = max(float(finite.min()), 1e-12)
        hi = float(finite.max())
        if hi / lo < 100.0:
            lo = 10.0 ** np.floor(np.log10(lo))
            hi = max(10.0 ** np.ceil(np.log10(hi)), lo * 100.0)
        norm = mpl.colors.LogNorm(vmin=lo, vmax=hi)
    im = ax.imshow(values, cmap=GRID_CMAP, norm=norm, origin="lower",
                   aspect="auto", interpolation=INTERPOLATION)
    ax.set_xticks(range(len(scales)))
    ax.set_xticklabels([f"{s:g}" for s in scales], fontsize=FONT_SIZE - 1)
    ax.set_yticks(range(len(crops)))
    ax.set_yticklabels([f"{c:g}" for c in crops], fontsize=FONT_SIZE - 1)
    ax.set_xlabel("Far-field samples per camera px", fontsize=FONT_SIZE)
    ax.set_ylabel("Fraction of the field seen", fontsize=FONT_SIZE)
    # Cells the loss could not be evaluated on are left blank rather than
    # coloured as if they had a value.
    for i, j in zip(*np.where(~np.isfinite(values))):
        ax.plot(j, i, "x", ms=3.0, color="0.5", mew=0.8)

    cax = _mm_axes(fig, fig_h_mm, xs[2] + FIG2_WIDTHS_MM[2] + CBAR_OFFSET_MM,
                   panel_y, CBAR_WIDTH_MM, FIG2_PANEL_MM)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=CBAR_FONT_SIZE)
    cbar.set_label("Projection NMSE, twin vs truth", fontsize=FONT_SIZE - 1,
                   labelpad=1)
    cbar.ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10.0, numticks=6))
    # Decade labels only; the minor ones are set in scientific notation and
    # push the colorbar's own label off the page.
    cbar.ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

    if path:
        fig.savefig(fig_path(path), dpi=FIG_DPI)
        print(f"Saved: {fig_path(path)}")
    return fig

In [ ]:
"""
Figure 1 - the full system.

Stray light present, the camera misaligned and fitted rather than known: the
hardest of the configurations, and the one that matches a real setup. The
ground truth is shown once here, so the later figures can go straight to the
discrepancy.
"""
_fov_um = first_order_um()
print(f"first order {_fov_um:.0f} um across, sampled {GEOMETRY.M} x {GEOMETRY.N}")
print(f"sensor spans {max(GEOMETRY.camera_shape) * GEOMETRY.camera_pixel_pitch:.0f} um"
      f" = {max(GEOMETRY.camera_shape) * GEOMETRY.camera_pixel_pitch / _fov_um:.3f} of it")

fit1 = run_fit(GEOMETRY, NUM_TRAIN_SAMPLES, TRUE_SEED, TWIN_SEED,
               use_background=True, ideal_affine=False, fix_camera=False,
               label="full")
cgh1 = cgh_check(fit1)
fig1 = figure_system(fit1, cgh1, blocks=("truth", "diff"), path="figS_full.pdf")

In [ ]:
"""
Figure 2 - the same optics under ideal measurement.

The camera perfectly aligned and held at the truth, no stray light:
everything the fit has to find is in the optics. The ground truth is not
drawn again - it is the same system, and what this figure adds is how much
closer the fit gets once the two hardest-to-observe parts of the model are
taken out of it.
"""
fit2 = run_fit(GEOMETRY, NUM_TRAIN_SAMPLES, TRUE_SEED, TWIN_SEED,
               use_background=False, ideal_affine=True, fix_camera=True,
               label="ideal")
cgh2 = cgh_check(fit2)
fig2 = figure_system(fit2, cgh2, blocks=("diff",), path="figS_ideal.pdf")

In [ ]:
"""
Figure 3 - fitting on a quarter of the field.

The system of figure 2, measured through a sensor that spans half the first
order in each direction - a quarter of it by area - sitting off-axis in one
quadrant. The same 200 x 200 array, at a finer pitch, so the loss is the
five-scale one the other figures use and the runs stay comparable.

The hologram is then designed and replayed across the whole far field: the
question is whether a model fitted on a corner still predicts the rest.

Two runs, because the answer depends on what the twin is allowed to
represent. Without the Seidel module it cannot describe a pupil aberration at
all, so whatever aberration the quadrant demands has to be absorbed by the
SLM field - a fit that is right where it was measured and wrong everywhere
else. With the module it can, and the aberration extrapolates.

Each run gets one frame and one letter: the parameters it recovered, its
convergence and its projection. The ground truth is not repeated.
"""
OFFAXIS_COVERAGE = 0.5        # linear, so a quarter of the field by area
OFFAXIS_CENTRE = (0.20, 0.20)
OFFAXIS_SHIFT_RANGE = 250.0   # px; the default 35 px clamp cannot reach

offaxis_geometry = geometry_with(
    camera_pixel_pitch=OFFAXIS_COVERAGE * first_order_um() / max(GEOMETRY.camera_shape),
    affine_shift_range=OFFAXIS_SHIFT_RANGE,
)
print(f"sensor spans {OFFAXIS_COVERAGE:.2f} of the order "
      f"({OFFAXIS_COVERAGE ** 2:.0%} by area), centred at {OFFAXIS_CENTRE}")

quarter = []
for pupil in (False, True):
    fit = run_fit(offaxis_geometry, NUM_TRAIN_SAMPLES, TRUE_SEED, TWIN_SEED,
                  use_background=False, ideal_affine=True, fix_camera=True,
                  twin_pupil=pupil, camera_offset=OFFAXIS_CENTRE,
                  label="with Seidel" if pupil else "no Seidel")
    quarter.append((fit, cgh_check(fit)))

fig3 = figure_runs(quarter, path="figS_quarter.pdf")

In [ ]:
"""
The study behind the statistics figure.

    samples    one system, six training-set sizes
    exposures  the same system at seven exposures
    grid       a 2D geometry sweep: how much of the field the sensor sees,
               against how coarsely it samples it

One ground truth throughout: TRUE_SEED for every run, so nothing here is
confounded by a different system being drawn. The grid is the one exception
worth naming - it uses a square SLM so that both of its axes are isotropic,
which is a different array from the one above but the same seed and the same
draw of scalars.

Results are written to JSON; the figure cell reads that and never retrains.
This is the expensive cell - the grid alone is 30 trainings, and the finely
sampled corner of it has an 800 x 800 sensor.

An "epoch" is one batch_size draw, not a pass over the data, so the step
budget is the same for every training-set size and the smallest set is also
the most heavily re-visited, as in the experimental runs.

STUDY_CONFIG applies throughout: no stray light, camera ideal and held at the
truth. The first two figures already measure what those cost, and holding
them fixed here means the map shows what the sweep costs rather than where
the correspondence stage happens to fail on an undersampled sensor.
"""
import json
from pathlib import Path

RESULTS_PATH = Path("sim_study.json")

STUDY_CONFIG = dict(use_background=False, ideal_affine=True, fix_camera=True)

SAMPLE_COUNTS = (10, 15, 20, 50, 100, 200)

# Exposure, applied by scaling the ground truth's SLM field. The camera model
# carries no photon or read noise, so the only thing an exposure can change
# is how much of the frame clips at saturation - which is what this sweep
# measures.
EXPOSURES = (0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0)

# --- the 2D geometry sweep -------------------------------------------------
# A square array, so the far field is square too and both axes of the sweep
# are isotropic.
GRID_SLM = (200, 200)
GRID_SAMPLES = 200

# Sensor width as a fraction of the first diffraction order, centred on it.
CROP_RATIOS = (0.2, 0.4, 0.6, 0.8, 1.0)
# Far-field samples per camera pixel. Above 1 the sensor is coarser than the
# model's own grid; below 1 it oversamples it.
PIXEL_SCALES = (0.5, 1.0, 2.0, 4.0, 6.0, 10.0)

# The two axes are independent by construction:
#     camera_pixel_pitch = p * far-field sample spacing
#     n_camera           = crop * N / p
# so changing the sensor size leaves the pitch alone, and vice versa.

# One MS-SSIM scale for the whole grid. Five scales cannot be built on an
# image shorter than 160 px and most of the grid is smaller than that; using
# one loss everywhere matters more than matching the rest of the study, whose
# losses are not comparable with these anyway.
GRID_SCALES = 1


def _record(fit, **extra):
    history = fit.history
    out = {
        "train": [float(v) for v in history["eval: structure"]],
        "validation_min": [float(v) for v in history["validation: structure (min)"]],
        "validation_max": [float(v) for v in history["validation: structure (max)"]],
        "final_train": float(history["eval: structure"][-1]),
    }
    out.update(extra)
    return out


study = {"samples": [], "exposures": [], "grid": []}

# --- how much data ---------------------------------------------------------
for n in SAMPLE_COUNTS:
    print(f"[samples] N = {n}", flush=True)
    fit = run_fit(GEOMETRY, n, TRUE_SEED, TWIN_SEED, **STUDY_CONFIG)
    study["samples"].append(_record(
        fit, num_train_samples=n,
        exposures_per_pair=ITERATIONS * BATCH_SIZE / n))

# --- how much exposure -----------------------------------------------------
for k in EXPOSURES:
    print(f"[exposure] {k}x", flush=True)
    fit = run_fit(GEOMETRY, NUM_TRAIN_SAMPLES, TRUE_SEED, TWIN_SEED,
                  slm_field_scale=k, **STUDY_CONFIG)
    study["exposures"].append(_record(
        fit, exposure=k,
        saturated=float((fit.saturation_mask == 0).float().mean())))

# --- the 2D geometry sweep -------------------------------------------------
_grid_base = geometry_with(slm_pixels=GRID_SLM)
_grid_delta = first_order_um(_grid_base) / _grid_base.N   # sample spacing, um

for crop in CROP_RATIOS:
    for p in PIXEL_SCALES:
        pitch = p * _grid_delta
        n_px = int(round(crop * _grid_base.N / p))
        print(f"[grid] crop {crop:.1f}, p {p:g} -> {n_px} px at {pitch:.2f} um",
              flush=True)
        cell = {"crop": crop, "pixel_scale": p, "camera_px": n_px,
                "camera_pitch_um": pitch}
        if n_px < 12:
            # The SSIM window is 11 px wide and is applied without padding.
            print("    sensor too small for the loss window; skipped", flush=True)
            cell["projection"] = None
            study["grid"].append(cell)
            continue
        geometry = geometry_with(slm_pixels=GRID_SLM, camera_shape=(n_px, n_px),
                                 camera_pixel_pitch=pitch)
        try:
            fit = run_fit(geometry, GRID_SAMPLES, TRUE_SEED, TWIN_SEED,
                          num_scales=GRID_SCALES, **STUDY_CONFIG)
            # cgh_check works on the far field, so every cell is scored on
            # the same grid however different its sensor.
            cgh = cgh_check(fit)
        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {str(exc)[:120]}", flush=True)
            cell["projection"] = None
            study["grid"].append(cell)
            continue
        cell["projection"] = cgh["truth_vs_twin"]["NMSE"]
        cell["final_train"] = float(fit.history["eval: structure"][-1])
        study["grid"].append(cell)
        print(f"    projection NMSE {cell['projection']:.3e}", flush=True)

RESULTS_PATH.write_text(json.dumps(study, indent=1))
print(f"Saved: {RESULTS_PATH}")

In [ ]:
"""
The statistics figure. Reads sim_study.json, so it rebuilds without
retraining anything.

    a  held-out loss against training-set size
    b  held-out loss against exposure
    c  the geometry sweep: how much of the field the sensor sees against how
       coarsely it samples it
"""
import json
from pathlib import Path

study = json.loads(Path("sim_study.json").read_text())

for name, key, unit in (("samples", "num_train_samples", "pairs"),
                        ("exposures", "exposure", "x")):
    finals = [(r[key], r["final_train"]) for r in study[name]]
    print(f"{name}: " + ", ".join(f"{v:g}{unit} {f:.2e}" for v, f in finals))

fig4 = figure_statistics(study, path="figS_statistics.pdf")